# SP Registry Manager

Notebook for inspecting and managing Storage Provider registrations.

## Operations covered
- Inspect all registered providers
- Register a new provider (`registerProviderFor`)
- Update available space (`updateAvailableSpace`)
- Update SLI capabilities (`setCapabilities`)
- Update price (`setPrice`)
- Pause / unpause a provider
- Block / unblock a provider (admin only)
- Update payee address

### Prerequisites
Run `00_setup.sh` through `02_deploy.sh` so the `.env` has contract addresses.

Addresses and keys are loaded automatically from:
`~/Forked/filecoin-boost/scripts/porep-market/.env`

## 0. Setup

In [ ]:
import builtins as _builtins
import json
import logging
import os
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from web3 import Web3
from web3.exceptions import ContractCustomError
from web3.middleware import ExtraDataToPOAMiddleware

# ── .env ──────────────────────────────────────────────────────────────────────
BOOST_ENV = Path.home() / "Forked/filecoin-boost/scripts/porep-market/.env"
if BOOST_ENV.exists():
    load_dotenv(BOOST_ENV)
    print(f"Loaded .env from {BOOST_ENV}")
else:
    load_dotenv()
    print(f"WARNING: {BOOST_ENV} not found")

# ── CONFIG ────────────────────────────────────────────────────────────────────
_key = os.getenv("PRIVATE_KEY_TEST", os.getenv("PRIVATE_KEY", ""))
CONFIG = {
    "rpc_url":     os.getenv("RPC_URL", "http://127.0.0.1:1234/rpc/v1"),
    "admin_key":   _key,
    "sp_registry": os.getenv("SP_REGISTRY", ""),
}
assert CONFIG["sp_registry"], "SP_REGISTRY not set — run 02_deploy.sh first"
CONFIG["sp_registry"] = Web3.to_checksum_address(CONFIG["sp_registry"])

print("\n── CONFIG ──────────────────────────────────────")
for _k, _v in CONFIG.items():
    print(f"  {_k:<30} {_v}")

print("\n── CONFIG ──────────────────────────────────────")
for _k, _v in CONFIG.items():
    print(f"  {_k:<30} {_v}")

# ── ABI dir ───────────────────────────────────────────────────────────────────
_nb = globals().get("__vsc_ipynb_file__", "")
ABI_DIR = Path(_nb).parent.parent / "abis" if _nb else BOOST_ENV.parent / "porep-market" / "abis"
assert ABI_DIR.exists(), f"ABI dir not found: {ABI_DIR}"
print(f"ABI dir : {ABI_DIR}")

def load_abi(name: str):
    with open(ABI_DIR / f"{name}.json") as f:
        return json.load(f)

# ── Logger ────────────────────────────────────────────────────────────────────
_log_dir = Path(_nb).parent / "logs" if _nb else Path.home() / "porep-market-logs"
_log_dir.mkdir(parents=True, exist_ok=True)
if not hasattr(_builtins, "_porep_spreg_log"):
    _builtins._porep_spreg_log = _log_dir / f"sp-registry-{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
LOG_FILE = _builtins._porep_spreg_log

logger = logging.getLogger("porep.spreg")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    _fh = logging.FileHandler(LOG_FILE, mode="a")
    _fh.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-5s  %(message)s"))
    _ch = logging.StreamHandler()
    _ch.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(_fh)
    logger.addHandler(_ch)
log  = logger.info
logw = logger.warning
loge = logger.error
log(f"=== Session log: {LOG_FILE} ===")

# ── Web3 ──────────────────────────────────────────────────────────────────────
w3 = Web3(Web3.HTTPProvider(CONFIG["rpc_url"]))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)
assert w3.is_connected(), f"Cannot connect to {CONFIG['rpc_url']}"
chain_id = w3.eth.chain_id
log(f"Connected  chain_id={chain_id}  latest_block={w3.eth.block_number}")

# ── Account ───────────────────────────────────────────────────────────────────
admin_acct = w3.eth.account.from_key(CONFIG["admin_key"]) if CONFIG["admin_key"] else None
log(f"Admin : {admin_acct.address if admin_acct else 'NOT SET'}")

# ── Contract ──────────────────────────────────────────────────────────────────
sp_registry = w3.eth.contract(address=CONFIG["sp_registry"], abi=load_abi("SPRegistry"))
log(f"SPRegistry at {CONFIG['sp_registry']}")

# ── Error map + helpers ───────────────────────────────────────────────────────
_error_map: dict[str, str] = {}
for _abi_file in ABI_DIR.glob("*.json"):
    try:
        for _entry in json.load(open(_abi_file)):
            if _entry.get("type") == "error":
                _sig = _entry["name"] + "(" + ",".join(i["type"] for i in _entry.get("inputs", [])) + ")"
                _error_map["0x" + w3.keccak(text=_sig).hex()[:8]] = _sig
    except Exception:
        pass

def decode_custom_error(data: str) -> str:
    return _error_map.get(data[:10], f"unknown error {data[:10]}")

def send_tx(fn, account, value=0):
    nonce = w3.eth.get_transaction_count(account.address)
    log(f"  → {fn.fn_name}  from={account.address}  nonce={nonce}")
    try:
        tx = fn.build_transaction({
            "from": account.address,
            "nonce": nonce,
            "gasPrice": w3.eth.gas_price,
            "value": value,
            "chainId": chain_id,
        })
    except ContractCustomError as e:
        err = decode_custom_error(e.data)
        loge(f"  ✗ estimate_gas reverted: {err}")
        raise RuntimeError(f"estimate_gas reverted: {err}") from e
    signed = account.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    log(f"  ⏳ sent tx={tx_hash.hex()}")
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    if receipt["status"] == 1:
        log(f"  ✅ success  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
    else:
        loge(f"  ❌ reverted  tx={tx_hash.hex()}  gas_used={receipt['gasUsed']}")
        raise RuntimeError(f"Transaction reverted: {tx_hash.hex()}")
    return receipt

def fmt_provider(actor_id, info):
    org, payee, paused, blocked, caps, avail, committed, pending, price = info
    status = "BLOCKED" if blocked else ("PAUSED" if paused else "active")
    log(f"Provider {actor_id} [{status}]")
    log(f"  organization : {org}")
    log(f"  payee        : {payee}")
    log(f"  available    : {avail:,} bytes  ({avail / 2**30:.2f} GiB)")
    log(f"  committed    : {committed:,} bytes")
    log(f"  pending      : {pending:,} bytes")
    log(f"  price/mo     : {price}  (0 = manual approval)")
    log(f"  capabilities : retrievability={caps[0]} bps  bandwidth={caps[1]} Mbps  latency={caps[2]} ms  indexing={caps[3]}%")

log(f"Setup complete. {len(_error_map)} error selectors loaded.")

---
## 1. Inspect all providers

In [ ]:
try:
    providers = sp_registry.functions.getProviders().call()
except Exception:
    providers = []
log(f"Registered providers ({len(providers)}): {providers}")

In [ ]:
def fmt_provider(actor_id, info):
    org, payee, paused, blocked, caps, avail, committed, pending, price = info
    status = "BLOCKED" if blocked else ("PAUSED" if paused else "active")
    log(f"Provider {actor_id} [{status}]")
    log(f"  organization : {org}")
    log(f"  payee        : {payee}")
    log(f"  available    : {avail:,} bytes  ({avail / 2**30:.1f} GiB)")
    log(f"  committed    : {committed:,} bytes")
    log(f"  pending      : {pending:,} bytes")
    log(f"  price/mo     : {price}  (0 = manual approval)")
    log(f"  capabilities : retrievability={caps[0]} bps  bandwidth={caps[1]} Mbps  latency={caps[2]} ms  indexing={caps[3]}%")

for actor_id in providers:
    info = sp_registry.functions.getProviderInfo(actor_id).call()
    fmt_provider(actor_id, info)
    log("")

---
## 2. Register a new provider

`registerProviderFor(actorId, organization, capabilities, availableBytes, pricePerSectorPerMonth, payee)`

Requires `OPERATOR_ROLE` or `DEFAULT_ADMIN_ROLE` — on devnet the deployer has both.

In [ ]:
# ── Parameters — edit before running ──────────────────────────────────────────
REG = {
    "actor_id":               1000,
    "organization":           admin_acct.address,   # org wallet
    "payee":                  admin_acct.address,   # payment recipient
    "retrievability_bps":     10_000,               # 100 %
    "bandwidth_mbps":         1_000,
    "latency_ms":             100,
    "indexing_pct":           100,
    "available_gib":          1,                    # GiB of storage offered
    "price_per_sector_month": 0,                    # 0 = manual deal approval
}

already = sp_registry.functions.isProviderRegistered(REG["actor_id"]).call()
log(f"Actor {REG['actor_id']} already registered: {already}")

In [ ]:
assert not already, f"Actor {REG['actor_id']} is already registered — use update cells below"
assert admin_acct, "Set PRIVATE_KEY_TEST"

capabilities = (
    REG["retrievability_bps"],
    REG["bandwidth_mbps"],
    REG["latency_ms"],
    REG["indexing_pct"],
)
available_bytes = REG["available_gib"] * 1024 ** 3

receipt = send_tx(
    sp_registry.functions.registerProviderFor(
        REG["actor_id"],
        REG["organization"],
        capabilities,
        available_bytes,
        REG["price_per_sector_month"],
        REG["payee"],
    ),
    admin_acct,
)

In [ ]:
registered = sp_registry.functions.isProviderRegistered(REG["actor_id"]).call()
log(f"Actor {REG['actor_id']} registered: {registered}")
info = sp_registry.functions.getProviderInfo(REG["actor_id"]).call()
fmt_provider(REG["actor_id"], info)

---
## 3. Update available space

`updateAvailableSpace(actorId, availableBytes)` — callable by provider controller or admin.

In [ ]:
UPDATE_ACTOR_ID  = 1000
NEW_AVAILABLE_GIB = 10   # new total available space in GiB

info = sp_registry.functions.getProviderInfo(UPDATE_ACTOR_ID).call()
log(f"Current available : {info[5]:,} bytes  ({info[5] / 2**30:.2f} GiB)")

In [ ]:
receipt = send_tx(
    sp_registry.functions.updateAvailableSpace(UPDATE_ACTOR_ID, NEW_AVAILABLE_GIB * 1024 ** 3),
    admin_acct,
)

In [ ]:
info = sp_registry.functions.getProviderInfo(UPDATE_ACTOR_ID).call()
log(f"Updated available : {info[5]:,} bytes  ({info[5] / 2**30:.2f} GiB)")

---
## 4. Update SLI capabilities

`setCapabilities(actorId, (retrievabilityBps, bandwidthMbps, latencyMs, indexingPct))`

In [ ]:
CAPS_ACTOR_ID = 1000
NEW_CAPS = {
    "retrievability_bps": 9_500,   # 95 %
    "bandwidth_mbps":     500,
    "latency_ms":         200,
    "indexing_pct":       80,
}

info = sp_registry.functions.getProviderInfo(CAPS_ACTOR_ID).call()
caps = info[4]
log(f"Current caps : retrievability={caps[0]} bps  bandwidth={caps[1]} Mbps  latency={caps[2]} ms  indexing={caps[3]}%")

In [ ]:
receipt = send_tx(
    sp_registry.functions.setCapabilities(
        CAPS_ACTOR_ID,
        (NEW_CAPS["retrievability_bps"], NEW_CAPS["bandwidth_mbps"], NEW_CAPS["latency_ms"], NEW_CAPS["indexing_pct"]),
    ),
    admin_acct,
)

In [ ]:
info = sp_registry.functions.getProviderInfo(CAPS_ACTOR_ID).call()
caps = info[4]
log(f"Updated caps : retrievability={caps[0]} bps  bandwidth={caps[1]} Mbps  latency={caps[2]} ms  indexing={caps[3]}%")

---
## 5. Update price

`setPrice(actorId, pricePerSectorPerMonth)` — 0 means manual deal approval.

In [ ]:
PRICE_ACTOR_ID = 1000
NEW_PRICE = 1_000_000   # MockUSDC units (6 decimals) per 32 GiB sector per month

info = sp_registry.functions.getProviderInfo(PRICE_ACTOR_ID).call()
log(f"Current price : {info[8]}")

In [ ]:
receipt = send_tx(
    sp_registry.functions.setPrice(PRICE_ACTOR_ID, NEW_PRICE),
    admin_acct,
)

In [ ]:
info = sp_registry.functions.getProviderInfo(PRICE_ACTOR_ID).call()
log(f"Updated price : {info[8]}")

---
## 6. Pause / unpause provider

In [ ]:
PAUSE_ACTOR_ID = 1000

info = sp_registry.functions.getProviderInfo(PAUSE_ACTOR_ID).call()
log(f"Provider {PAUSE_ACTOR_ID} paused={info[2]}  blocked={info[3]}")

In [ ]:
# Toggle: pause if active, unpause if paused
if not info[2]:  # not paused
    receipt = send_tx(sp_registry.functions.pauseProvider(PAUSE_ACTOR_ID), admin_acct)
else:
    receipt = send_tx(sp_registry.functions.unpauseProvider(PAUSE_ACTOR_ID), admin_acct)

In [ ]:
info = sp_registry.functions.getProviderInfo(PAUSE_ACTOR_ID).call()
log(f"Provider {PAUSE_ACTOR_ID} paused={info[2]}  blocked={info[3]}")

---
## 7. Block / unblock provider  _(admin only)_

In [ ]:
BLOCK_ACTOR_ID = 1000

info = sp_registry.functions.getProviderInfo(BLOCK_ACTOR_ID).call()
log(f"Provider {BLOCK_ACTOR_ID} blocked={info[3]}")

In [ ]:
# Toggle: block if not blocked, unblock if blocked
if not info[3]:  # not blocked
    receipt = send_tx(sp_registry.functions.blockProvider(BLOCK_ACTOR_ID), admin_acct)
else:
    receipt = send_tx(sp_registry.functions.unblockProvider(BLOCK_ACTOR_ID), admin_acct)

In [ ]:
info = sp_registry.functions.getProviderInfo(BLOCK_ACTOR_ID).call()
log(f"Provider {BLOCK_ACTOR_ID} blocked={info[3]}")

---
## 8. Committed providers snapshot

In [ ]:
committed_providers = sp_registry.functions.getCommittedProviders().call()
log(f"Providers with committed capacity ({len(committed_providers)}): {committed_providers}")

In [ ]:
log("=" * 55)
log("FULL REGISTRY SNAPSHOT")
log("=" * 55)
all_providers = sp_registry.functions.getProviders().call()
for actor_id in all_providers:
    info = sp_registry.functions.getProviderInfo(actor_id).call()
    fmt_provider(actor_id, info)
    log("")
log(f"Total: {len(all_providers)} provider(s)")